# Eine Mini-Mini-Deep-Learning-Bibliothek für MLPs

In `computing_the_gradient.ipynb` kannte jeder skalare `Node` seine Operation
und deren lokale Ableitungsregel. In `computing_the_gradient_mlp.ipynb` haben
wir dieselbe Idee auf Vektoren und Matrizen angewendet. Jetzt verpacken wir
diese Rechnungen in **kleine, lesbare Schichtklassen**.

Unsere Bibliothek besteht aus `Linear`, `ReLU`, `Sigmoid`, dem Loss `MSE`
und einer Schichtenliste `Sequential`. Wir benötigen nur **NumPy**.
Führe die Zellen von oben nach unten aus.

Wir betrachten weiterhin **ein Trainingsbeispiel pro Vorwärts- und Rückwärtspass**.
Eingaben, Ausgaben und Bias sind zweidimensionale Spaltenvektoren `(n, 1)`.
Ein MLP ist hier eine Kette: Statt eines allgemeinen Graphen genügt eine Liste
in Berechnungsreihenfolge. Jeder Eintrag dieser Liste ist jetzt eine ganze Schicht.

## 1. Eine gemeinsame Idee für alle Schichten

Jede Schicht stellt zwei Methoden bereit:

- `forward(x)` berechnet die Ausgabe und merkt sich die benötigten Werte.
- `backward(grad_output)` erhält die **Adjungierte der Ausgabe** und gibt die
  **Adjungierte der Eingabe** zurück. Bei `Linear` speichert sie zusätzlich die Parametergradienten.

Für $v=f(u)$ bedeutet das: `grad_output` ist $\bar v=\partial L/\partial v$,
der Rückgabewert ist $\bar u=\partial L/\partial u$.
Die lokale Ableitung von $f$ und die Adjungierte sind unterschiedliche Größen:
**Erst die Kettenregel verbindet sie.**

Wir bauen keine große Jacobi-Matrix auf. Die Rückwärtsregeln berechnen direkt,
wie der ankommende Gradient weitergereicht wird.

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 2. Linear: die drei bekannten Rückwärtsregeln

Für $v=Wu+b$ gilt:

$$\bar W=\bar v\,u^T,\qquad \bar b=\bar v,\qquad \bar u=W^T\bar v.$$

`W` hat Shape `(n_outputs, n_inputs)`, `b` hat Shape `(n_outputs, 1)`.
Wir speichern die Eingabe in `self.x`, weil wir sie rückwärts für $\bar W$ brauchen.
`grad_W` und `grad_b` sind die Adjungierten der Parameter.
Der Name „Linear“ ist in DL-Bibliotheken üblich; mit Bias ist die Abbildung **affin**.

In [2]:
class Linear:
    def __init__(self, n_inputs, n_outputs, rng):
        # Zufällige Gewichte brechen die Symmetrie zwischen den Neuronen.
        self.W = rng.normal(0.0, 1 / np.sqrt(n_inputs),
                            size=(n_outputs, n_inputs))
        self.b = np.zeros((n_outputs, 1))

    def forward(self, x):
        self.x = x                       # Für den Rückwärtspass merken.
        return self.W @ x + self.b

    def backward(self, grad_output):
        self.grad_W = grad_output @ self.x.T
        self.grad_b = grad_output.copy()
        grad_input = self.W.T @ grad_output
        return grad_input

    def step(self, learning_rate):
        # Erst nach dem vollständigen Rückwärtspass aufrufen!
        self.W -= learning_rate * self.grad_W
        self.b -= learning_rate * self.grad_b

## 3. ReLU: nur aktive Neuronen lassen den Gradienten durch

$$v_i=\max(0,u_i),\qquad \bar u=\bar v\odot(u>0).$$

`*` multipliziert elementweise. Bei $u_i=0$ ist ReLU nicht differenzierbar;
wir legen dort für den Rückwärtspass den Wert 0 fest.
Diese Schicht besitzt keine trainierbaren Parameter.

In [3]:
class ReLU:
    def forward(self, x):
        self.x = x
        return np.maximum(0.0, x)

    def backward(self, grad_output):
        return grad_output * (self.x > 0)

## 4. Sigmoid: die logistische Aktivierungsfunktion

$$v_i=\sigma(u_i)=\frac{1}{1+e^{-u_i}},\qquad
\sigma'(u_i)=v_i(1-v_i).$$

Deshalb merken wir uns hier die **Ausgabe** und berechnen rückwärts:

$$\bar u=\bar v\odot v\odot(1-v).$$

Sigmoid bildet auf Werte zwischen 0 und 1 ab. Bei sehr großen Beträgen der
Eingabe wird die Ableitung klein. Die zwei Fälle im Vorwärtscode vermeiden,
dass die Exponentialfunktion bei großen negativen Eingaben überläuft;
beide Fälle berechnen dieselbe mathematische Funktion.

In [4]:
class Sigmoid:
    def forward(self, x):
        self.output = np.empty_like(x, dtype=float)
        positive = (x >= 0)
        self.output[positive] = 1 / (1 + np.exp(-x[positive]))
        exp_x = np.exp(x[~positive])
        self.output[~positive] = exp_x / (1 + exp_x)
        return self.output

    def backward(self, grad_output):
        return grad_output * self.output * (1 - self.output)

## 5. MSE: vom skalaren Verlust zurück zur Vorhersage

Für $m$ Ausgabewerte eines Trainingsbeispiels verwenden wir den **Mean Squared Error**:

$$L=\frac{1}{m}\sum_{i=1}^{m}(\hat y_i-y_i)^2,\qquad
\frac{\partial L}{\partial\hat y_i}=\frac{2}{m}(\hat y_i-y_i).$$

**Achtung zum vorigen Notebook:** Dort war $L=\tfrac12(\hat y-y)^2$.
Hier verwenden wir den MSE **ohne Faktor ½**. Für eine einzelne Ausgabe
sind deshalb Loss und Gradienten doppelt so groß wie dort.

`forward(prediction, target)` liefert eine Zahl. `backward()` startet mit
$\bar L=1$ und liefert den Spaltenvektor $\bar{\hat y}$.
Das Ziel behandeln wir als feste Vorgabe; wir berechnen keinen Zielgradienten.

In [5]:
class MSE:
    def forward(self, prediction, target):
        # Gleiche Shapes verhindern ungewolltes NumPy-Broadcasting.
        if prediction.shape != target.shape:
            raise ValueError("Vorhersage und Ziel müssen dieselbe Shape haben.")
        self.error = prediction - target
        return np.mean(self.error ** 2).item()

    def backward(self, grad_loss=1.0):
        return grad_loss * 2 * self.error / self.error.size

## 6. Das bekannte 3 → 2 → 1-Netz: jeden Aufruf sehen

Wir setzen dieselben Parameter wie im MLP-Notebook ein.
Die Zufallsinitialisierung wird hier durch die bekannten Zahlen ersetzt.

```text
x ──→ Linear 1 ──→ z ──→ ReLU ──→ h ──→ Linear 2 ──→ ŷ ──→ MSE(ŷ, y) ──→ L
  ←── grad_x  ←── grad_z  ←────── grad_h  ←────────── grad_y_hat  ←────────── 1
```

Zuerst der Vorwärtspass: **Werte von links nach rechts**.

In [6]:
rng = np.random.default_rng(42)
linear1 = Linear(3, 2, rng)
relu = ReLU()
linear2 = Linear(2, 1, rng)
loss_function = MSE()

linear1.W = np.array([[1.0, 0.0, -1.0], [-1.0, 1.0, 1.0]])
linear1.b = np.array([[1.0], [1.0]])
linear2.W = np.array([[2.0, -1.0]])
linear2.b = np.array([[0.5]])

x = np.array([[1.0], [2.0], [-1.0]])
y = np.array([[4.0]])

z = linear1.forward(x)
h = relu.forward(z)
y_hat = linear2.forward(h)
loss = loss_function.forward(y_hat, y)

for name, value in [("x", x), ("z", z), ("h", h), ("ŷ", y_hat)]:
    print(f"{name} (Shape {value.shape}):\n{value}")
print("MSE =", loss)

x (Shape (3, 1)):
[[ 1.]
 [ 2.]
 [-1.]]
z (Shape (2, 1)):
[[3.]
 [1.]]
h (Shape (2, 1)):
[[3.]
 [1.]]
ŷ (Shape (1, 1)):
[[5.5]]
MSE = 2.25


Jetzt der Rückwärtspass: **Adjungierte von rechts nach links**.
Jede Zeile erhält genau den Gradienten, den die nachfolgende Operation zurückgibt.

In [7]:
grad_y_hat = loss_function.backward()      # Start mit ∂L/∂L = 1.
grad_h = linear2.backward(grad_y_hat)
grad_z = relu.backward(grad_h)
grad_x = linear1.backward(grad_z)

for name, gradient in [("ŷ", grad_y_hat), ("h", grad_h),
                       ("z", grad_z), ("x", grad_x)]:
    print(f"∂L/∂{name} =\n{gradient}")

for name, layer in [("Linear 1", linear1), ("Linear 2", linear2)]:
    print(f"\n{name}:\n∂L/∂W =\n{layer.grad_W}\n∂L/∂b =\n{layer.grad_b}")

# Kontrolle mit den bekannten Zahlen (jetzt ohne den Faktor ½ im Loss).
np.testing.assert_allclose(loss, 2.25)
np.testing.assert_allclose(linear1.grad_W, [[6, 12, -6], [-3, -6, 3]])
np.testing.assert_allclose(linear2.grad_W, [[9, 3]])
np.testing.assert_allclose(grad_x, [[9], [-3], [-9]])

∂L/∂ŷ =
[[3.]]
∂L/∂h =
[[ 6.]
 [-3.]]
∂L/∂z =
[[ 6.]
 [-3.]]
∂L/∂x =
[[ 9.]
 [-3.]
 [-9.]]

Linear 1:
∂L/∂W =
[[ 6. 12. -6.]
 [-3. -6.  3.]]
∂L/∂b =
[[ 6.]
 [-3.]]

Linear 2:
∂L/∂W =
[[9. 3.]]
∂L/∂b =
[[3.]]


### Was wurde gerade automatisiert?

Der Vorwärtspass liefert $\hat y=5{,}5$ und $L=2{,}25$.
Rückwärts startet der MSE mit $\bar{\hat y}=2(5{,}5-4)=3$.
Linear 2 reicht $\bar h=(6,-3)^T$ zurück. Beide ReLU-Eingaben sind positiv,
also ist $\bar z=\bar h$. Linear 1 berechnet daraus seine Parametergradienten
und $\bar x=(9,-3,-9)^T$.

Anders als bei den skalaren `Node`-Objekten verteilen wir hier die Gradienten
nicht über allgemeine Vorgängerlisten. In dieser Kette reicht jede Schicht
ihren Eingabegradienten direkt an die vorige Schicht zurück.
Die Beiträge mehrerer Neuronen werden in `W.T @ grad_output` summiert.

Wir **überschreiben** `grad_W` und `grad_b` pro Rückwärtspass.
Darum benötigen wir für diese Bibliothek kein `zero_grad()`.
Gradienten mehrerer Beispiele werden hier nicht angesammelt.

## 7. Sequential: zwei Schleifen statt vieler einzelner Aufrufe

Der Mechanismus bleibt gleich: vorwärts die Liste durchlaufen,
rückwärts `reversed(layers)` durchlaufen.
`step()` verändert nur die Parameter der `Linear`-Schichten.

In [8]:
class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad_output):
        for layer in reversed(self.layers):
            grad_output = layer.backward(grad_output)
        return grad_output

    def step(self, learning_rate):
        for layer in self.layers:
            if isinstance(layer, Linear):
                layer.step(learning_rate)

In [9]:
# Dieselben Objekte und Parameter wie oben: nur die Verkettung ist jetzt kürzer.
model = Sequential([linear1, relu, linear2])

prediction = model.forward(x)
loss_before = loss_function.forward(prediction, y)
grad_x = model.backward(loss_function.backward())

# Erst jetzt, nach dem gesamten Rückwärtspass, die Gewichte verändern.
model.step(learning_rate=0.01)

prediction_after = model.forward(x)
loss_after = loss_function.forward(prediction_after, y)
print(f"Ein Lernschritt: MSE {loss_before:.6f} → {loss_after:.6f}")

Ein Lernschritt: MSE 2.250000 → 0.022952


**Wichtig für diese bewusst kleine Implementierung:** Jede Schicht merkt sich
nur die Werte ihres **letzten** Vorwärtspasses. Es gilt daher immer:
`forward → Loss → backward → step`. Zwischen dem zugehörigen Vorwärts- und
Rückwärtspass dürfen wir keinen anderen Vorwärtspass einschieben oder Parameter ändern.
Für jede Position im Netz erzeugen wir ein eigenes Schichtobjekt; gemeinsame
Schichtinstanzen und verzweigte Graphen sind hier nicht vorgesehen.

Der Eingabegradient ist lehrreich, aber beim üblichen Training ändern wir nur
die Gewichte und Biaswerte, nicht die Eingaben oder Zielwerte.

## 8. Stimmen die Rückwärtsregeln? Eine numerische Kontrolle

Wir prüfen ein kleines Netz **mit beiden Aktivierungsfunktionen und zwei Ausgaben**.
Damit wird auch der Mittelwertfaktor $1/m$ im MSE geprüft.
Die ReLU-Eingaben liegen von 0 entfernt, damit wir keine Knickstelle differenzieren.

Für jeden Parameter vergleichen wir `backward()` mit der zentralen Differenz:

$$\frac{\partial L}{\partial\theta}\approx
\frac{L(\theta+\varepsilon)-L(\theta-\varepsilon)}{2\varepsilon}.$$

Das ist eine Kontrolle, kein Trainingsverfahren: Sie benötigt zwei Vorwärtspässe
pro Eintrag. Auch den zurückgereichten Eingabegradienten prüfen wir.

In [10]:
rng_check = np.random.default_rng(12)
first = Linear(3, 2, rng_check)
last = Linear(2, 2, rng_check)
first.W = np.array([[0.2, -0.3, 0.4], [-0.5, 0.1, 0.2]])
first.b = np.array([[0.8], [-0.8]])
check_model = Sequential([first, ReLU(), last, Sigmoid()])
check_loss = MSE()
check_x = np.array([[0.4], [-0.7], [0.2]])
check_y = np.array([[0.2], [0.8]])


def current_loss():
    return check_loss.forward(check_model.forward(check_x), check_y)


current_loss()
analytic_x = check_model.backward(check_loss.backward())

# Kopien sichern: Die folgenden Vorwärtspässe überschreiben die Zwischenspeicher.
checks = [("W1", first.W, first.grad_W.copy()),
          ("b1", first.b, first.grad_b.copy()),
          ("W2", last.W, last.grad_W.copy()),
          ("b2", last.b, last.grad_b.copy()),
          ("x", check_x, analytic_x.copy())]

epsilon = 1e-6
for name, values, analytic in checks:
    numeric = np.zeros_like(values)
    for index in np.ndindex(values.shape):
        original = values[index]
        values[index] = original + epsilon
        loss_plus = current_loss()
        values[index] = original - epsilon
        loss_minus = current_loss()
        values[index] = original
        numeric[index] = (loss_plus - loss_minus) / (2 * epsilon)

    np.testing.assert_allclose(analytic, numeric, rtol=1e-5, atol=1e-8)
    print(f"{name}: bestanden, größte Abweichung {np.max(np.abs(analytic - numeric)):.2e}")

W1: bestanden, größte Abweichung 1.74e-11
b1: bestanden, größte Abweichung 2.90e-12
W2: bestanden, größte Abweichung 3.64e-12
b2: bestanden, größte Abweichung 9.04e-12
x: bestanden, größte Abweichung 1.44e-11


## 9. Ein kleines MLP wirklich trainieren: XOR

XOR liefert 1, wenn genau eine der beiden Eingaben 1 ist, sonst 0.
Wir verwenden `Linear → ReLU → Linear → Sigmoid` und MSE.
Sigmoid begrenzt die Ausgabe auf Werte zwischen 0 und 1.

Wir durchlaufen die vier Trainingsbeispiele wiederholt. Ein vollständiger
Durchlauf heißt **Epoche**. Nach **jedem einzelnen Beispiel** folgt ein
Lernschritt (stochastischer Gradientenabstieg, hier in fester Reihenfolge).
`reshape(-1, 1)` macht aus einer Datenzeile einen Spaltenvektor.

In [11]:
X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
Y = np.array([[0.0], [1.0], [1.0], [0.0]])

# Fester Seed: Das Beispiel ist beim vollständigen Neustart reproduzierbar.
rng_train = np.random.default_rng(42)
xor_model = Sequential([
    Linear(2, 8, rng_train),
    ReLU(),
    Linear(8, 1, rng_train),
    Sigmoid(),
])
xor_loss = MSE()
learning_rate = 0.1


def evaluate_xor():
    # Nur auswerten; danach beginnt das nächste Training mit einem neuen forward().
    predictions = np.array([xor_model.forward(row.reshape(-1, 1)).item()
                            for row in X])
    mse = np.mean((predictions - Y[:, 0]) ** 2).item()
    return predictions, mse


_, initial_mse = evaluate_xor()
print(f"Vor dem Training: MSE = {initial_mse:.6f}")

for epoch in range(1, 3001):
    for row_x, row_y in zip(X, Y):
        sample_x = row_x.reshape(-1, 1)
        sample_y = row_y.reshape(-1, 1)

        prediction = xor_model.forward(sample_x)      # 1. Werte vorwärts
        loss = xor_loss.forward(prediction, sample_y) # 2. Verlust berechnen
        grad_prediction = xor_loss.backward()         # 3. Adjungierte der Ausgabe
        xor_model.backward(grad_prediction)           # 4. Adjungierte rückwärts
        xor_model.step(learning_rate)                 # 5. Parameter ändern

    if epoch in [1, 100, 500, 1000, 3000]:
        _, mse = evaluate_xor()
        print(f"Epoche {epoch:4d}: MSE = {mse:.6f}")

predictions, final_mse = evaluate_xor()
print("\nEingabe       Ziel   Vorhersage   Klasse (Schwelle 0.5)")
for row, target, prediction in zip(X, Y[:, 0], predictions):
    print(f"{row}    {target:.0f}      {prediction:.4f}        {int(prediction >= 0.5)}")

Vor dem Training: MSE = 0.252503
Epoche    1: MSE = 0.251265
Epoche  100: MSE = 0.122772
Epoche  500: MSE = 0.005833
Epoche 1000: MSE = 0.001993
Epoche 3000: MSE = 0.000478

Eingabe       Ziel   Vorhersage   Klasse (Schwelle 0.5)
[0. 0.]    0      0.0372        0
[0. 1.]    1      0.9851        1
[1. 0.]    1      0.9858        1
[1. 1.]    0      0.0104        0


## 10. Die Bibliothek auf einen Blick

| Baustein | Vorwärts | Rückwärts |
|---|---|---|
| `Linear` | $v=Wu+b$ | $\bar W=\bar v u^T$, $\bar b=\bar v$, Rückgabe $W^T\bar v$ |
| `ReLU` | $v=\max(0,u)$ | $\bar v\odot(u>0)$ |
| `Sigmoid` | $v=1/(1+e^{-u})$ | $\bar v\odot v\odot(1-v)$ |
| `MSE` | $L=\frac1m\sum_i(\hat y_i-y_i)^2$ | $\bar{\hat y}=\frac2m(\hat y-y)$ bei $\bar L=1$ |
| `Sequential` | Schichten von vorne nach hinten | Schichten von hinten nach vorne |

**Merksatz:** Jede Schicht kennt nur ihren eigenen Vorwärts- und Rückwärtsschritt.
Die Schichtenliste reicht die Werte vorwärts und die Adjungierten rückwärts weiter.
Erst danach aktualisieren wir $W$ und $b$ mit `parameter -= learning_rate * gradient`.

Zum Ausprobieren: Verändere die Zahl der versteckten Neuronen oder ersetze
die versteckte `ReLU()` durch eine **neue** `Sigmoid()`-Instanz. Starte die
Trainingszelle jeweils neu und beobachte, wie sich die Lernkurve verändert.
Andere Startgewichte oder Lernraten können zu anderen Ergebnissen führen;
insbesondere können ReLU-Neuronen inaktiv bleiben und keine Gradienten weiterreichen.

## 11. Kompakte Graphvariante: Verzweigungen erlauben

Wir verwenden `Linear`, `ReLU`, `Sigmoid` und `MSE` von oben weiter.
Statt `Sequential` bekommt jetzt jeder **Graphknoten** eine Schicht und eine
Liste seiner direkten Vorgänger – wie beim skalaren `Node`, nur mit Arrays.
Zum Zusammenführen zweier Zweige ergänzen wir die Rechenoperation `Add`.

```text
                                    ┌──→ Linear A ──┐
x ──→ Linear gemeinsam ──→ ReLU (h) ─┤              ├──→ Add ──→ Sigmoid ──→ MSE
                                    └──→ Linear B ──┘
```

Die entscheidenden Regeln:

- **Zusammenführen vorwärts:** Für $s=a+b$ liefert `Add.backward()` rückwärts
  $(\bar s,\bar s)$, denn beide lokalen Ableitungen sind 1. Der Gradient wird
  an beide Eingänge weitergereicht, nicht halbiert.
- **Verzweigen vorwärts:** Bei `h` kommen rückwärts zwei Beiträge an.
  Deshalb gilt $\bar h=W_A^T\bar a+W_B^T\bar b$ und im Code **`parent.bar += grad`**.
- Die Liste `graph` enthält jeden Knoten **genau einmal**, und zwar nach seinen
  Vorgängern (**topologische Reihenfolge**). Beim umgekehrten Durchlauf sind
  alle Beiträge eines Knotens angekommen, bevor er selbst rückwärts rechnet.

Jede Schichtinstanz gehört weiterhin zu genau einem Graphknoten. Den **Wert
eines Knotens** dürfen beliebig viele Nachfolger verwenden. Ein gemeinsamer
Linear-Knoten erhält dadurch den summierten Gradienten beider Wege.


In [12]:
class Add:
    def forward(self, a, b):
        # Hier nur gleich große Arrays: kein Broadcasting.
        if a.shape != b.shape:
            raise ValueError("Add benötigt zwei Arrays gleicher Shape.")
        return a + b

    def backward(self, grad_output):
        return grad_output, grad_output


class GraphNode:
    def __init__(self, name, layer=None, inputs=(), value=None):
        self.name = name
        self.layer = layer
        self.inputs = inputs
        self.value = value

    def forward(self):
        if self.layer is not None:
            values = [parent.value for parent in self.inputs]
            self.value = self.layer.forward(*values)
        # Jeder neue Vorwärtspass beginnt mit leeren Adjungierten.
        self.bar = np.zeros_like(self.value)

    def backward(self):
        if self.layer is None:            # Eingabeknoten: nichts weiterreichen.
            return
        gradients = self.layer.backward(self.bar)
        if len(self.inputs) == 1:
            gradients = (gradients,)      # Ein Gradient pro Vorgängerknoten.
        for parent, grad in zip(self.inputs, gradients):
            parent.bar += grad           # Beiträge ALLER Nachfolger sammeln!


In [13]:
# Netz aufbauen: h wird von zwei verschiedenen Knoten verwendet.
rng_graph = np.random.default_rng(42)
gx = GraphNode("x", value=np.array([[1.0], [2.0]]))
gz = GraphNode("z", Linear(2, 2, rng_graph), inputs=(gx,))
gh = GraphNode("h", ReLU(), inputs=(gz,))
ga = GraphNode("a", Linear(2, 1, rng_graph), inputs=(gh,))
gb = GraphNode("b", Linear(2, 1, rng_graph), inputs=(gh,))
gs = GraphNode("s = a + b", Add(), inputs=(ga, gb))
gp = GraphNode("ŷ", Sigmoid(), inputs=(gs,))
graph = [gx, gz, gh, ga, gb, gs, gp]      # Jeden Knoten genau einmal aufführen.
graph_loss = MSE()
graph_target = np.array([[0.8]])

# Kleine feste Zahlen, damit die beiden Wege leicht nachzurechnen sind.
gz.layer.W = np.array([[1.0, 0.5], [-0.5, 1.0]])
gz.layer.b = np.array([[0.1], [0.2]])
ga.layer.W = np.array([[0.2, -0.1]])
gb.layer.W = np.array([[-0.1, 0.3]])

# 1. Vorwärts: Werte berechnen und Adjungierte auf 0 setzen.
print("Vorwärts:")
for node in graph:
    node.forward()
    print(f"{node.name}: {node.value.ravel()}")
graph_loss_before = graph_loss.forward(gp.value, graph_target)

# 2. Rückwärts: MSE startet mit ∂L/∂L = 1; danach alle Beiträge aufsummieren.
gp.bar = graph_loss.backward()
print("\nRückwärts (jeweils vor dem Weiterreichen):")
for node in reversed(graph):
    print(f"∂L/∂({node.name}): {node.bar.ravel()}")
    node.backward()

# Genau diese beiden Beiträge sind bei h zusammengekommen.
contribution_a = ga.layer.W.T @ ga.bar
contribution_b = gb.layer.W.T @ gb.bar
print("\nBeitrag über A:", contribution_a.ravel())
print("Beitrag über B:", contribution_b.ravel())
print("Summe bei h:  ", gh.bar.ravel())
np.testing.assert_allclose(gh.bar, contribution_a + contribution_b)
print("Gradient der gemeinsamen Gewichte:\n", gz.layer.grad_W)

# 3. Lernschritt: jeden Linear-Knoten genau einmal aktualisieren.
for node in graph:
    if isinstance(node.layer, Linear):
        node.layer.step(learning_rate=0.1)

# 4. Erneut vorwärts, um den Effekt des Lernschritts zu sehen.
for node in graph:
    node.forward()
graph_loss_after = graph_loss.forward(gp.value, graph_target)
print(f"\nEin Lernschritt: MSE {graph_loss_before:.6f} → {graph_loss_after:.6f}")


Vorwärts:
x: [1. 2.]
z: [2.1 1.7]
h: [2.1 1.7]
a: [0.25]
b: [0.3]
s = a + b: [0.55]
ŷ: [0.6341]

Rückwärts (jeweils vor dem Weiterreichen):
∂L/∂(ŷ): [-0.3317]
∂L/∂(s = a + b): [-0.077]
∂L/∂(b): [-0.077]
∂L/∂(a): [-0.077]
∂L/∂(h): [-0.0077 -0.0154]
∂L/∂(z): [-0.0077 -0.0154]
∂L/∂(x): [-0.     -0.0192]

Beitrag über A: [-0.0154  0.0077]
Beitrag über B: [ 0.0077 -0.0231]
Summe bei h:   [-0.0077 -0.0154]
Gradient der gemeinsamen Gewichte:
 [[-0.0077 -0.0154]
 [-0.0154 -0.0308]]

Ein Lernschritt: MSE 0.027511 → 0.018539


**Merksatz:** Ein Knoten darf vorwärts mehrere Nachfolger haben.
Rückwärts **addiert** er deren Gradientenbeiträge und reicht erst dann die
gesamte Adjungierte weiter. Die Rückwärtsregeln der Schichten bleiben gleich;
neu sind die Vorgängerliste, `+=` und die passende Reihenfolge.

Die Graphvariante ersetzt `Sequential` für dieses Beispiel. Sie benötigt einen
Graphen ohne Zyklen und eine von uns korrekt angegebene topologische Reihenfolge.
Wie zuvor gehört zu jedem Rückwärtspass ein frischer Vorwärtspass; mehrere
Lernschritte entstehen durch Wiederholen von Vorwärtslauf, Loss, Rückwärtslauf
und Parameterupdate. Die abschließende Auswertung setzt hier die Adjungierten
bereits wieder auf 0; die angezeigten Gradienten stammen aus dem Lauf davor.
